In [11]:
import pandas as pd
import json
from api_caller import call_api

In [12]:
df = pd.read_csv("data/EuroParl/scored_llama/full.csv")

In [13]:
def extract_prompt(text):
    user_prompt = f"""You are a social science research assistant specializing in political discourse analysis. Your goal is to transform ideological statements into "Neutral Policy Interrogatives."

Task:
Analyze the provided European Parliament transcript and convert the ideological speech into five (5) distinct, open-ended prompts. Each prompt should be designed to elicit a normative opinion from an LLM regarding the specific conflict mentioned in the text.

Instructions:
1. Identify the Core Conflict: Pinpoint the specific stance or tension mentioned in the speech.
2. Remove Bias: Strip away the speaker's emotional adjectives and predetermined conclusions.
3. Formulate 5 Diverse Prompts: 
    - Each prompt must ask for a normative judgment (a "should" or "ought" question).
    - Vary the focus of each prompt (e.g., one focusing on economic impact, one on human rights/ethics, one on sovereignty/governance, etc.).
4. Ensure Specificity: The prompts must be granular enough to prevent the LLM from providing a generic or "both-sides" canned response.

Output Format (Strict JSON):
{{
  "extracted_topic": "...",
  "llm_prompts": [
    "Prompt 1...",
    "Prompt 2...",
    "Prompt 3...",
    "Prompt 4...",
    "Prompt 5..."
  ]
}}

Text:
{text}
"""
    response = None
    while True:
      try:
        response = call_api(user_prompt)
        result = json.loads(response["choices"][0]["message"]["content"])
        if len(result) == 1:
          return result
        print(f"returned only {len(result)} instead of 1 response")
      except Exception as e:
        print(f"{e} thrown for {text}, {response}")
      except:
        print(f"Failed for other reason on {text}, repeating")


In [14]:
def process_df(df, ifrom=0, istep=500):
    total_len = len(df)
    j = ifrom
    
    while j < total_len:
        kend = min(j + istep, total_len)
        try:
            df_subset = df.iloc[j:kend].copy()
            extracted_topics = []
            llm_prompts_list = []

            for i, row in df_subset.iterrows():
                progress = (j + (df_subset.index.get_loc(i))) / total_len * 100
                print(f"Progress: {progress:.2f}%")
                
                text_content = row["en"]
                result = extract_prompt(text_content)
                extracted_topics.append(result.get("extracted_topic", ""))
                llm_prompts_list.append(result.get("llm_prompts", []))
            
            df_subset["extracted_topic"] = extracted_topics
            df_subset["llm_prompt"] = llm_prompts_list
            
            final_df = df_subset[["en", "llm_prompt", "extracted_topic", "EU Party"]]
            output_path = f"data/EuroParl/extracted_prompts_llama/extracted_prompts_{j}_{kend}.csv"
            
            final_df.to_csv(
                output_path,
                sep=";",
                index=False,
                encoding="utf-8-sig"
            )
            
            print(f"Saved index {j} to {kend}")
            j += istep
            
        except Exception as e:
            print(f"Error at index {j}: {e}")

In [15]:
process_df(df.iloc[:10], ifrom=0)

Progress: 0.00%
Expecting value: line 1 column 1 (char 0) thrown for Last weekend in Ireland the announcement by Dell, that it was relocating 2 000 jobs, came as a body-blow to the community in the Mid-West and West of Ireland. In this context, the European Globalisation Fund could prove to be especially important to help retrain and reskill workers and to assist in the promotion of entrepreneurship for self-employment. It is crucial that the Irish Government makes an immediate application to the Globalisation Fund, so that workers can have some faith in the future and see that the EU is working to assist all workers and, in this case, those in the West and Mid-West of Ireland., {'id': 'llama3.3:latest-9efcf105-9e8e-4acb-83da-2fc07a43e0e3', 'created': 1772393897, 'model': 'llama3.3:latest', 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'message': {'role': 'assistant', 'content': '```json\n{\n  "extracted_topic": "Dell\'s relocation of 2000 jobs from Ireland and po